In [1]:
pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd

df = pd.read_csv("../data/processed/dataset_final.csv")


In [7]:
df

,transaction_type_online purchase,transaction_type_transfer,transaction_type_withdrawal,merchant_category_Clothing,merchant_category_Crypto,merchant_category_Electronics,merchant_category_Entertainment,merchant_category_Food,merchant_category_Gambling,merchant_category_Services,...,account_age_days,description_risk_score,transaction_frequency_24h,num_failed_transactions_7d,transaction_hour,is_weekend,num_devices_used,num_countries_used,kyc_verified,is_fraud
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,-0.929790,-0.368841,0.000000,0.000000,8.000000,0.000000,2.000000,0.000000,0.000000,0
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.761706,-0.368841,0.000000,0.000000,7.000000,0.000000,2.000000,0.000000,1.000000,0
2,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.964482,-0.368841,0.000000,0.000000,19.000000,0.000000,1.000000,0.000000,0.000000,1
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.560900,3.310755,0.000000,0.000000,3.000000,0.000000,1.000000,1.000000,1.000000,1
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.064750,-0.368841,0.000000,0.000000,10.000000,0.000000,2.000000,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-0.203948,-0.750026,1.910238,-1.254917,17.590397,1.515989,1.451670,-0.632398,2.499171,0
1196,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.703249,-0.775283,-0.428285,0.797061,8.393788,0.931623,0.287628,-0.664923,0.176913,0
1197,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-1.015343,3.217018,-0.875200,0.716758,2.408601,2.285924,2.401097,0.323237,1.236132,1
1198,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,-1.476658,-0.078490,-0.208216,-0.946535,17.659267,0.274012,2.283382,-0.844388,0.658808,0


In [ ]:
X = df.drop(columns=["is_fraud"])

y = df["is_fraud"]



In [38]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report
)


In [29]:
xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1
)


In [37]:
from sklearn.metrics import classification_report, roc_auc_score
xgb_base.fit(X_train,y_train)
y_pred = xgb_base.predict(X_test)
y_proba = xgb_base.predict_proba(X_test)[:, 1]



print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("")
print(classification_report(y_test, y_pred))


Accuracy: 0.825
Precision: 0.7083333333333334
Recall: 0.3269230769230769
F1 Score: 0.4473684210526316
ROC AUC: 0.6254091653027823

              precision    recall  f1-score   support

           0       0.84      0.96      0.90       188
           1       0.71      0.33      0.45        52

    accuracy                           0.82       240
   macro avg       0.77      0.64      0.67       240
weighted avg       0.81      0.82      0.80       240



In [23]:
param_grid = {
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 3, 5],
    "gamma": [0, 0.1, 0.3],
    "subsample": [0.8, 0.9],
    "colsample_bytree": [0.8, 0.9],
    "reg_alpha": [0, 0.1, 1],
    "reg_lambda": [1, 1.5, 2]
}


In [24]:
grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring="f1",      # fraud-aware
    cv=5,
    verbose=1,
    n_jobs=-1
)


In [25]:
grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 972 candidates, totalling 4860 fits


GridSearchCV(cv=5,
             estimator=XGBClassifier(base_score=None, booster=None,
                                     callbacks=None, colsample_bylevel=None,
                                     colsample_bynode=None,
                                     colsample_bytree=None, device=None,
                                     early_stopping_rounds=None,
                                     enable_categorical=False,
                                     eval_metric='logloss', feature_types=None,
                                     feature_weights=None, gamma=None,
                                     grow_policy=None, importance_type=None,
                                     interaction_constraint...
                                     max_leaves=None, min_child_weight=None,
                                     missing=nan, monotone_constraints=None,
                                     multi_strategy=None, n_estimators=300,
                                     n_jobs=-1, num_parallel_tree=None, ...),
             n_jobs=-1,
             param_grid={'colsample_bytree': [0.8, 0.9], 'gamma': [0, 0.1, 0.3],
                         'max_depth': [3, 4, 5], 'min_child_weight': [1, 3, 5],
                         'reg_alpha': [0, 0.1, 1], 'reg_lambda': [1, 1.5, 2],
                         'subsample': [0.8, 0.9]},
             scoring='f1', verbose=1)

In [45]:
best_xgb = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

y_pred = best_xgb.predict(X_test)
y_proba = best_xgb.predict_proba(X_test)[:, 1]

from sklearn.metrics import classification_report, roc_auc_score





print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))
print("")
print(classification_report(y_test, y_pred))



Best Parameters:
{'colsample_bytree': 0.8, 'gamma': 0, 'max_depth': 3, 'min_child_weight': 1, 'reg_alpha': 0, 'reg_lambda': 1.5, 'subsample': 0.8}
Accuracy: 0.8458333333333333
Precision: 0.8947368421052632
Recall: 0.3269230769230769
F1 Score: 0.4788732394366197
ROC AUC: 0.6507774140752864

              precision    recall  f1-score   support

           0       0.84      0.99      0.91       188
           1       0.89      0.33      0.48        52

    accuracy                           0.85       240
   macro avg       0.87      0.66      0.69       240
weighted avg       0.85      0.85      0.82       240



In [39]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [46]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [42]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}


In [47]:
cv_results = cross_validate(
    estimator=best_xgb,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)


In [48]:
import numpy as np

for metric in cv_results:
    if metric.startswith("test_"):
        scores = cv_results[metric]
        print(
            f"{metric.replace('test_', '').upper():10s}: "
            f"{scores.mean():.3f} ± {scores.std():.3f}"
        )


ACCURACY  : 0.852 ± 0.018
PRECISION : 0.873 ± 0.069
RECALL    : 0.375 ± 0.094
F1        : 0.517 ± 0.093
ROC_AUC   : 0.691 ± 0.062


Baseline Model

Accuracy: 0.83

Precision (fraud): 0.71

Recall (fraud): 0.33

F1-score (fraud): 0.45

ROC-AUC: 0.63

Interpretation:
The baseline model shows moderate fraud detection capability. While overall accuracy is reasonable, the model misses many fraud cases (low recall) and has only moderate balance between precision and recall.

Fine-Tuned Model (GridSearchCV)

Accuracy: 0.85 ⬆

Precision (fraud): 0.89 ⬆

Recall (fraud): 0.33 (unchanged)

F1-score (fraud): 0.48 ⬆

ROC-AUC: 0.65 ⬆

Interpretation:
Hyperparameter tuning improved overall performance. Precision increased significantly, meaning fewer false positives, while recall remained unchanged. Higher F1-score and ROC-AUC indicate better generalization and discrimination compared to the baseline.

Cross-Validation Results (Model Robustness)

Accuracy  : 0.852 ± 0.018
Precision : 0.873 ± 0.069
Recall    : 0.375 ± 0.094
F1        : 0.517 ± 0.093
ROC-AUC   : 0.691 ± 0.062

Interpretation:

Performance is stable across folds (low standard deviation for accuracy).

Recall and F1-score improve under cross-validation, confirming the model performs better across different data splits.

ROC-AUC close to 0.70 shows meaningful predictive signal beyond random guessing.

Variability in recall is expected due to class imbalance.

Overall Conclusion (Best Answer for Assessment)

The fine-tuned XGBoost model outperforms the baseline model in terms of accuracy, precision, F1-score, and ROC-AUC. Cross-validation confirms that the model generalizes well and provides stable performance across multiple folds. Although fraud recall remains moderate, the model prioritizes high precision, making it suitable for conservative fraud detection where minimizing false positives is important.